In [80]:
!pip install -Uqq ucimlrepo
!pip install -Uqq fastbook

In [81]:
from fastbook import *
import fastbook
fastbook.setup_book

<function fastbook.setup_book()>

In [82]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
real_estate_valuation = fetch_ucirepo(id=477) 
  
# data (as pandas dataframes) 
X = real_estate_valuation.data.features 
y = real_estate_valuation.data.targets 
  
# metadata 
#print(f"Metadata: ({real_estate_valuation.metadata})\n\n") 
  
# variable information 
#print(f"Variables: ({real_estate_valuation.variables})\n\n") 
print(f"Columns: {X.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 6 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   X1 transaction date                     414 non-null    float64
 1   X2 house age                            414 non-null    float64
 2   X3 distance to the nearest MRT station  414 non-null    float64
 3   X4 number of convenience stores         414 non-null    int64  
 4   X5 latitude                             414 non-null    float64
 5   X6 longitude                            414 non-null    float64
dtypes: float64(5), int64(1)
memory usage: 19.5 KB
Columns: None


In [83]:
from fastai.data.transforms import RandomSplitter
trn_split,val_split=RandomSplitter(seed=42)(X)

In [84]:
trn_indep, val_indep = X.iloc[trn_split], X.iloc[val_split]
trn_dep, val_dep = y.iloc[trn_split], y.iloc[val_split]
trn_indep.shape, trn_dep.shape, val_indep.shape, val_dep.shape

((332, 6), (332, 1), (82, 6), (82, 1))

In [85]:
trn_indep.head()

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude
290,2013.083,37.7,490.3446,0,24.97217,121.53471
383,2012.667,29.1,506.1144,4,24.97845,121.53889
226,2013.000,16.5,4082.0150,0,24.94155,121.50381
29,2013.083,7.1,451.2438,5,24.97563,121.54694
59,2013.083,13.3,336.0532,5,24.95776,121.53438


In [86]:
from fastai.imports import *

# Step 1: Create time features and normalize all independent variables

In [87]:
from sklearn import preprocessing
import numpy as np

def preprocess_independent_variables(indpt_vs, scaler = None):
    time_df = pd.DataFrame(index=indpt_vs.index)
    
    fractional, integral = np.modf(indpt_vs["X1 transaction date"])
    time_df["year"] = pd.Series(integral.astype(int))
    time_df["month"] = pd.Series(np.round(fractional*12))

    columns = ['X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude']
    indpt_vs_columns_df = pd.concat([time_df, indpt_vs[columns]], axis=1)
    
    min_max_scaler = scaler
    if min_max_scaler is None:
        s = preprocessing.MinMaxScaler()
        min_max_scaler = s.fit(indpt_vs_columns_df)
    indpt_vs_columns_minmax = min_max_scaler.transform(indpt_vs_columns_df)

    columns = ['year_norm', 'month_norm', "house_age",'distance_to_nearest_MRT_station', 'number_of_convenience_stores', 'latitude', 'longitude']
    df = pd.DataFrame(
        indpt_vs_columns_minmax,
        columns=columns,
        index=indpt_vs.index
    )
    return df, min_max_scaler

In [88]:
trn_indep, scaler = preprocess_independent_variables(trn_indep)
trn_indep.head()

,year_norm,month_norm,house_age,distance_to_nearest_MRT_station,number_of_convenience_stores,latitude,longitude
290,1.0,0.090909,0.860731,0.072233,0.0,0.485943,0.659694
383,0.0,0.727273,0.664384,0.074673,0.4,0.562046,0.704766
226,1.0,0.000000,0.376712,0.627820,0.0,0.114881,0.326504
29,1.0,0.090909,0.162100,0.066185,0.5,0.527872,0.791568
59,1.0,0.090909,0.303653,0.048366,0.5,0.311318,0.656135


# Step 2: Turn X and Y into tensors

In [89]:
idpnt_column_names = ['year_norm', 'month_norm', 'house_age', 'distance_to_nearest_MRT_station',
       'number_of_convenience_stores', 'latitude', 'longitude']
n_columns = len(idpnt_column_names)
train_indpt = tensor(trn_indep[idpnt_column_names])
n_columns, train_indpt[:5]

(7,
 tensor([[1.0000, 0.0909, 0.8607, 0.0722, 0.0000, 0.4859, 0.6597],
         [0.0000, 0.7273, 0.6644, 0.0747, 0.4000, 0.5620, 0.7048],
         [1.0000, 0.0000, 0.3767, 0.6278, 0.0000, 0.1149, 0.3265],
         [1.0000, 0.0909, 0.1621, 0.0662, 0.5000, 0.5279, 0.7916],
         [1.0000, 0.0909, 0.3037, 0.0484, 0.5000, 0.3113, 0.6561]]))

In [90]:
train_dpt = tensor(trn_dep)
train_dpt[:5]

tensor([[37.0000],
        [40.3000],
        [12.8000],
        [57.1000],
        [42.4000]])

# Step 3: Train linear model

In [91]:
def init_coeffs():
    coeffs = torch.rand(n_columns+1,1) - 0.5
    coeffs[-1] = 0.0
    return coeffs.requires_grad_()

In [92]:
def calc_preds(coeffs, train_indpt):
    weights = coeffs[:-1]
    intercept = coeffs[-1]
    return train_indpt @ weights + intercept

In [93]:
def calc_loss(coeffs, train_indpt, train_dep):
    preds = calc_preds(coeffs, train_indpt)
    return torch.abs(preds-train_dep).mean()
    

In [94]:
def update_coeffs(coeffs, lr=0.01):
    coeffs.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()

In [95]:
def train_epoch(coeffs, lr=0.01):
    loss = calc_loss(coeffs, train_indpt, train_dpt)
    loss.backward()
    with torch.no_grad(): update_coeffs(coeffs, lr)
    print(f"{loss:.3}")

In [96]:
def train_model(epochs=30, lr=0.01):
    #torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): train_epoch(coeffs, lr)
    return coeffs

In [97]:
coeffs = train_model(epochs=100,lr=6)

38.3
22.1
10.6
9.4
9.14
8.91
8.69
8.49
8.3
8.14
7.97
7.81
7.65
7.5
7.35
7.2
7.07
6.97
6.89
6.82
6.77
6.72
6.7
6.65
6.62
6.58
6.55
6.52
6.5
6.47
6.46
6.43
6.41
6.41
6.38
6.42
6.42
6.42
6.42
6.41
6.39
6.38
6.38
6.38
6.43
6.34
6.34
6.34
6.39
6.32
6.36
6.32
6.43
6.36
6.49
6.33
6.42
6.34
6.47
6.34
6.47
6.33
6.46
6.33
6.48
6.37
6.5
6.34
6.52
6.34
6.51
6.34
6.5
6.31
6.43
6.36
6.47
6.32
6.47
6.34
6.45
6.35
6.43
6.32
6.41
6.34
6.41
6.33
6.41
6.33
6.39
6.35
6.41
6.33
6.42
6.32
6.41
6.33
6.41
6.32


In [98]:
def measure_loss(coeffs): 
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_loss(coeffs, tensor(ivs), tensor(val_dep))

lin_loss = measure_loss(coeffs)
lin_loss

tensor(7.2097, grad_fn=<MeanBackward0>)

In [99]:
def show_coeffs(): return dict(zip(idpnt_column_names, coeffs))
show_coeffs()

{'year_norm': tensor([4.1862], grad_fn=<UnbindBackward0>),
 'month_norm': tensor([4.7464], grad_fn=<UnbindBackward0>),
 'house_age': tensor([-12.1699], grad_fn=<UnbindBackward0>),
 'distance_to_nearest_MRT_station': tensor([-11.5518], grad_fn=<UnbindBackward0>),
 'number_of_convenience_stores': tensor([14.1175], grad_fn=<UnbindBackward0>),
 'latitude': tensor([19.7563], grad_fn=<UnbindBackward0>),
 'longitude': tensor([14.4777], grad_fn=<UnbindBackward0>)}

In [100]:
val_ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)

test_item = 0
pred = calc_preds(coeffs, tensor(val_ivs)[test_item])
val_indep.iloc[test_item], val_dep.iloc[test_item],pred

(X1 transaction date                       2012.75000
 X2 house age                                16.10000
 X3 distance to the nearest MRT station    4066.58700
 X4 number of convenience stores              0.00000
 X5 latitude                                 24.94297
 X6 longitude                               121.50342
 Name: 384, dtype: float64,
 Y house price of unit area    12.9
 Name: 384, dtype: float64,
 tensor([16.2680], grad_fn=<AddBackward0>))

# Step 4: Train an ensemble of linear models

In [101]:
def ensemble():
    coeffs = train_model(epochs=30,lr=7)
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_preds(coeffs, tensor(ivs))

In [102]:
learns = [ensemble() for _ in range(100)]

38.6
19.9
9.75
9.38
9.07
8.81
8.56
8.35
8.15
7.96
7.77
7.59
7.41
7.24
7.09
7.02
6.9
6.88
6.78
6.8
6.71
6.69
6.6
6.61
6.55
6.59
6.58
6.71
6.51
6.56
37.9
19.2
9.57
9.25
8.97
8.71
8.47
8.26
8.06
7.87
7.68
7.5
7.33
7.17
7.03
6.92
6.83
6.77
6.72
6.68
6.66
6.62
6.57
6.54
6.52
6.48
6.48
6.56
6.64
6.5
37.8
19.1
9.54
9.23
8.94
8.68
8.44
8.23
8.03
7.84
7.66
7.48
7.3
7.14
7.01
6.9
6.82
6.78
6.74
6.68
6.64
6.62
6.56
6.53
6.53
6.52
6.53
6.59
6.72
6.51
38.8
20.1
9.81
9.44
9.12
8.85
8.6
8.38
8.19
7.99
7.8
7.62
7.44
7.27
7.11
6.98
6.88
6.82
6.77
6.75
6.68
6.65
6.59
6.54
6.51
6.52
6.5
6.53
6.55
6.64
38.0
19.3
9.58
9.27
8.98
8.71
8.48
8.26
8.07
7.88
7.69
7.51
7.34
7.17
7.02
6.92
6.84
6.8
6.73
6.7
6.65
6.61
6.57
6.54
6.5
6.49
6.46
6.44
6.52
6.62
38.6
19.9
9.73
9.36
9.06
8.79
8.55
8.33
8.13
7.93
7.75
7.56
7.39
7.22
7.06
6.96
6.87
6.82
6.76
6.73
6.66
6.62
6.57
6.54
6.5
6.49
6.47
6.46
6.52
6.62
37.5
18.9
9.68
9.37
9.08
8.81
8.56
8.35
8.15
7.95
7.77
7.58
7.41
7.24
7.09
6.96
6.87
6.8
6.74
6.71
6.66
6.63
6.59


In [103]:
ens_preds = torch.stack(learns).mean(0)

In [104]:
lin_ens_loss = torch.abs(ens_preds-tensor(val_dep)).mean()
lin_ens_loss

tensor(6.8811, grad_fn=<MeanBackward0>)

In [105]:
# Calculate MAE separately for each existing linear-model prediction.
with torch.no_grad():
    val_targets = tensor(val_dep)
    lin_member_maes = torch.stack([
        torch.abs(preds - val_targets).mean()
        for preds in learns
    ])

# Save the summary before "learns" is reused for neural networks.
lin_ensemble_summary = {
    "Model family": "Linear",
    "Mean individual MAE": lin_member_maes.mean().item(),
    "Lowest individual MAE": lin_member_maes.min().item(),
    "Ensemble MAE": lin_ens_loss.item(),
}

pd.DataFrame([lin_ensemble_summary]).round(4)

,Model family,Mean individual MAE,Lowest individual MAE,Ensemble MAE
0,Linear,7.0455,6.7146,6.8811


# Step 5: Train neural net

In [106]:
def init_coeffs(n_hidden = 10):
    layer1 = (torch.rand(n_columns, n_hidden)-0.5).requires_grad_()
    layer2 = (torch.rand(n_hidden, 1)-.3).requires_grad_()
    const = torch.rand(1)[0].requires_grad_()
    return layer1, layer2, const

In [107]:
import torch.nn.functional as F

def calc_preds(coeffs, indpts):
    l1, l2, c = coeffs
    r = F.relu(indpts@l1)
    return r@l2 + c

In [108]:
def update_coeffs(coeffs, lr):
    for c in coeffs:
        c.sub_(c.grad * lr)
        c.grad.zero_()

In [109]:
coeffs = train_model(epochs=70,lr=1e-1)

37.3
37.0
36.6
36.1
35.6
34.9
34.2
33.2
32.0
30.5
28.7
26.4
23.5
19.9
15.9
12.1
9.88
9.26
9.12
9.0
8.9
8.8
8.7
8.61
8.52
8.43
8.36
8.29
8.21
8.14
8.07
8.01
7.95
7.89
7.82
7.76
7.71
7.65
7.59
7.53
7.47
7.41
7.35
7.29
7.24
7.19
7.15
7.12
7.08
7.05
7.02
6.99
6.96
6.94
6.91
6.89
6.87
6.85
6.83
6.81
6.8
6.78
6.76
6.75
6.73
6.72
6.71
6.7
6.69
6.68


In [110]:
def measure_loss(coeffs): 
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_loss(coeffs, tensor(ivs), tensor(val_dep))

nn_loss = measure_loss(coeffs)
nn_loss

tensor(7.1803, grad_fn=<MeanBackward0>)

# Step 6: Train an ensemble of neural networks

In [111]:
def ensemble():
    coeffs = train_model(epochs=100,lr=1.5e-1)
    ivs, _ = preprocess_independent_variables(val_indep, scaler=scaler)
    return calc_preds(coeffs, tensor(ivs))

In [112]:
learns = [ensemble() for _ in range(100)]

37.3
36.9
36.5
36.0
35.4
34.7
33.7
32.4
30.6
28.0
24.5
19.7
14.0
10.2
9.4
9.21
9.03
8.86
8.7
8.54
8.39
8.25
8.12
7.99
7.87
7.77
7.66
7.57
7.47
7.38
7.29
7.21
7.14
7.07
7.01
6.95
6.9
6.86
6.83
6.8
6.77
6.74
6.71
6.69
6.67
6.65
6.63
6.62
6.61
6.6
6.59
6.57
6.56
6.55
6.54
6.53
6.52
6.51
6.5
6.49
6.48
6.47
6.46
6.45
6.44
6.43
6.42
6.42
6.41
6.4
6.39
6.38
6.38
6.37
6.36
6.36
6.34
6.33
6.33
6.32
6.31
6.3
6.29
6.28
6.27
6.26
6.26
6.25
6.25
6.25
6.23
6.22
6.22
6.22
6.21
6.2
6.19
6.2
6.19
6.2
37.6
37.3
37.0
36.7
36.2
35.7
34.9
33.9
32.5
30.6
28.0
24.2
19.2
13.4
9.93
9.41
9.22
9.04
8.87
8.7
8.55
8.41
8.28
8.15
8.03
7.92
7.82
7.72
7.63
7.53
7.44
7.35
7.27
7.18
7.11
7.05
6.99
6.94
6.89
6.85
6.82
6.79
6.76
6.74
6.71
6.69
6.67
6.65
6.63
6.62
6.6
6.59
6.58
6.57
6.56
6.54
6.54
6.53
6.52
6.51
6.5
6.49
6.48
6.47
6.47
6.46
6.45
6.44
6.44
6.43
6.42
6.42
6.41
6.4
6.4
6.39
6.38
6.38
6.37
6.37
6.36
6.36
6.35
6.35
6.34
6.34
6.33
6.33
6.32
6.32
6.31
6.31
6.3
6.3
6.3
6.29
6.29
6.29
6.29
6.28
36.8
36.2
35.5
34.5

In [113]:
ens_preds = torch.stack(learns).mean(0)

In [114]:
nn_ens_loss = torch.abs(ens_preds-tensor(val_dep)).mean()
nn_ens_loss

tensor(7.0507, grad_fn=<MeanBackward0>)

In [115]:
# Calculate MAE separately for each existing neural-network prediction.
with torch.no_grad():
    val_targets = tensor(val_dep)
    nn_member_maes = torch.stack([
        torch.abs(preds - val_targets).mean()
        for preds in learns
    ])

# Save the neural-network summary separately.
nn_ensemble_summary = {
    "Model family": "Neural network",
    "Mean individual MAE": nn_member_maes.mean().item(),
    "Lowest individual MAE": nn_member_maes.min().item(),
    "Ensemble MAE": nn_ens_loss.item(),
}

pd.DataFrame([nn_ensemble_summary]).round(4)

,Model family,Mean individual MAE,Lowest individual MAE,Ensemble MAE
0,Neural network,7.0721,6.8014,7.0507


# Loss Scores

## 5 models in an ensemble

In [77]:
print(f"Linear model: {lin_loss}")
print(f"Linear model (ens): {lin_ens_loss}")
print(f"Neural net: {nn_loss}")
print(f"Neural net (ens): {nn_ens_loss}")

Linear model: 7.209723472595215
Linear model (ens): 6.810535907745361
Neural net: 7.254626750946045
Neural net (ens): 7.084789752960205


## 100 models in an ensemble

In [116]:
print(f"Linear model: {lin_loss}")
print(f"Linear model (ens): {lin_ens_loss}")
print(f"Neural net: {nn_loss}")
print(f"Neural net (ens): {nn_ens_loss}")

Linear model: 7.209723472595215
Linear model (ens): 6.8811211585998535
Neural net: 7.180288791656494
Neural net (ens): 7.0507283210754395


In [117]:
available_summaries = [
    globals()[name]
    for name in ("lin_ensemble_summary", "nn_ensemble_summary")
    if name in globals()
]

ensemble_comparison = pd.DataFrame(available_summaries).round(4)
ensemble_comparison

,Model family,Mean individual MAE,Lowest individual MAE,Ensemble MAE
0,Linear,7.0455,6.7146,6.8811
1,Neural network,7.0721,6.8014,7.0507
